<a href="https://colab.research.google.com/github/D2718281828nis/Neuro-Neuron_Models/blob/master/models/example-Kuramoto-oscillators.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Kuramoto Oscillator vs Simple Sinusoidal Oscillator


This notebook demonstrates:
1. How simple sinusoidal oscillators behave independently
2. How Kuramoto oscillators synchronize through coupling
3. Key differences between uncoupled vs coupled oscillator systems

**Main Idea**: The Kuramoto model shows how a group of oscillators with different natural frequencies can synchronize through weak coupling - a phenomenon observed in fireflies, heart cells, power grids, and brain waves.

The unit circle is a key tool for visualizing phase relationships in oscillator systems.
- **Radius = 1**: All oscillators are represented on a circle of radius 1.
- **Angle = Phase (θ)**: The angle from the positive x‑axis represents the oscillator’s current phase.
- **Position = (cosθ, sinθ)**: Cartesian coordinates show the oscillator’s state.

How to Interpret It

1. **Single Oscillator Movement**  
   - A full circle (2π radians) = one complete oscillation cycle.  
   - Constant angular speed = natural frequency (ω).  
   - Faster movement = higher natural frequency.

2. **Synchronization Visualization**  
   - ✅ **Synchronized**: Points cluster together (similar angles).  
   - ❌ **Desynchronized**: Points spread around the circle.  
   - 🔄 **Partial sync**: Some clustering with occasional spreading.

3. **Order Parameter (r)**  
   The vector from the center to the “center of mass” of all points.  
   - **r = 0**: Complete desynchronization (evenly distributed points).  
   - **r = 1**: Perfect synchronization (all points at the same position).  
   - **0 < r < 1**: Partial synchronization.

4. **Phase Locking**  
   - Points maintain a fixed angular distance.  
   - Oscillators have synchronized frequencies but different phases.  
   - Common in the Kuramoto model below critical coupling.


Real‑World Analogy

Imagine dancers on a circular stage:  
- Each dancer moves at their own natural pace (natural frequency).  
- When they coordinate (coupling), they form groups moving together.  
- The tighter the group (higher `r`), the better the synchronization.  
- The center of the group shows the average rhythm of the system.

In [13]:
# Import required libraries
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import matplotlib.gridspec as gridspec
import ipywidgets as widgets
from IPython.display import display, clear_output
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg

In [14]:


# Kuramoto model definition
def kuramoto_model(t, theta, omega, K):
    """Kuramoto model differential equations"""
    N = len(omega)
    dtheta = np.zeros(N)
    for i in range(N):
        dtheta[i] = omega[i] + (K/N) * np.sum(np.sin(theta - theta[i]))
    return dtheta

# %% [code]
# Simulation parameters
omega = [2 * np.pi, 2.5 * np.pi]  # 1.0 Hz and 1.25 Hz
K_coupled = 1.8  # Subcritical coupling
t_span = [0, 5]
t_eval = np.linspace(0, 5, 1000)
theta0 = [0, 0]

# Simulate uncoupled oscillators
K_uncoupled = 0
sol_uncoupled = solve_ivp(lambda t, theta: kuramoto_model(t, theta, omega, K_uncoupled),
                          t_span, theta0, t_eval=t_eval)

# Simulate coupled oscillators
sol_coupled = solve_ivp(lambda t, theta: kuramoto_model(t, theta, omega, K_coupled),
                        t_span, theta0, t_eval=t_eval)

# Convert phases to actual signals (sinusoidal outputs)
signal_uncoupled = np.sin(sol_uncoupled.y)
signal_coupled = np.sin(sol_coupled.y)

# Calculate order parameter
def calculate_r(theta):
    return np.abs(np.sum(np.exp(1j * theta), axis=0)) / len(omega)

r_coupled = calculate_r(sol_coupled.y)
r_uncoupled = calculate_r(sol_uncoupled.y)

In [15]:

# Create the timeline control widget (completely separate from visualizations)
timeline_container = widgets.VBox()
time_slider = widgets.FloatSlider(
    value=0,
    min=0,
    max=5,
    step=0.01,
    description='Time (s):',
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='.2f'
)

play_button = widgets.Play(
    value=0,
    min=0,
    max=5,
    step=0.05,
    interval=100,
    description="► Play",
    disabled=False
)

# Link the play button to the slider
widgets.jslink((play_button, 'value'), (time_slider, 'value'))

# Create a separate output widget for the timeline
timeline_output = widgets.Output()

# Create a separate output widget for the main visualization
main_output = widgets.Output()

# %% [code]
# Function to create the visualization (without any UI elements)
def create_visualization(t):
    # Create a clean figure with no UI elements incorporated
    fig = plt.figure(figsize=(16, 10))
    gs = gridspec.GridSpec(2, 2, height_ratios=[1, 1.5], figure=fig)

    # Main plots
    ax_phase = fig.add_subplot(gs[0, :])
    ax_signal = fig.add_subplot(gs[1, 0], sharex=ax_phase)
    ax_polar = fig.add_subplot(gs[1, 1], projection='polar')

    # Initialize plots
    line_phase1_u, = ax_phase.plot(t_eval, sol_uncoupled.y[0], 'r-', alpha=0.5, label='Uncoupled θ₁ (1.0 Hz)')
    line_phase2_u, = ax_phase.plot(t_eval, sol_uncoupled.y[1], 'b-', alpha=0.5, label='Uncoupled θ₂ (1.25 Hz)')
    line_phase1_c, = ax_phase.plot(t_eval, sol_coupled.y[0], 'r-', linewidth=2.5, label='Coupled θ₁')
    line_phase2_c, = ax_phase.plot(t_eval, sol_coupled.y[1], 'b-', linewidth=2.5, label='Coupled θ₂')
    phase_marker1, = ax_phase.plot([], [], 'ro', markersize=12, zorder=5)
    phase_marker2, = ax_phase.plot([], [], 'bo', markersize=12, zorder=5)
    ax_phase.set_title('Phase Evolution Over Time', fontsize=16)
    ax_phase.set_ylabel('Phase (radians)', fontsize=14)
    ax_phase.grid(alpha=0.3)
    ax_phase.legend(loc='upper left', fontsize=12)
    ax_phase.set_ylim(-1, 35)
    ax_phase.set_xlim(0, 5)

    # Add reference lines for key phase positions
    ax_phase.axhline(y=0, color='k', linestyle=':', alpha=0.3)
    ax_phase.axhline(y=np.pi/2, color='g', linestyle=':', alpha=0.3)
    ax_phase.axhline(y=np.pi, color='k', linestyle=':', alpha=0.3)
    ax_phase.axhline(y=3*np.pi/2, color='g', linestyle=':', alpha=0.3)
    ax_phase.axhline(y=2*np.pi, color='k', linestyle=':', alpha=0.3)
    ax_phase.text(4.8, -1, '0', ha='right', color='k')
    ax_phase.text(4.8, np.pi/2-0.3, 'π/2', ha='right', color='g')
    ax_phase.text(4.8, np.pi-0.3, 'π', ha='right', color='k')
    ax_phase.text(4.8, 3*np.pi/2-0.3, '3π/2', ha='right', color='g')
    ax_phase.text(4.8, 2*np.pi-0.3, '2π', ha='right', color='k')

    line_signal1_u, = ax_signal.plot(t_eval, signal_uncoupled[0], 'r-', alpha=0.5, label='Uncoupled Signal 1')
    line_signal2_u, = ax_signal.plot(t_eval, signal_uncoupled[1], 'b-', alpha=0.5, label='Uncoupled Signal 2')
    line_signal1_c, = ax_signal.plot(t_eval, signal_coupled[0], 'r-', linewidth=2, label='Coupled Signal 1')
    line_signal2_c, = ax_signal.plot(t_eval, signal_coupled[1], 'b-', linewidth=2, label='Coupled Signal 2')
    signal_marker1, = ax_signal.plot([], [], 'ro', markersize=12, zorder=5)
    signal_marker2, = ax_signal.plot([], [], 'bo', markersize=12, zorder=5)
    ax_signal.set_title('Signal Output (sin(θ)) Over Time', fontsize=16)
    ax_signal.set_xlabel('Time (s)', fontsize=14)
    ax_signal.set_ylabel('Amplitude', fontsize=14)
    ax_signal.grid(alpha=0.3)
    ax_signal.legend(loc='upper left', fontsize=12)
    ax_signal.set_ylim(-1.1, 1.1)

    # Add reference lines for signal values
    ax_signal.axhline(y=0, color='k', linestyle=':', alpha=0.3)
    ax_signal.axhline(y=1, color='g', linestyle=':', alpha=0.3)
    ax_signal.axhline(y=-1, color='g', linestyle=':', alpha=0.3)

    # Polar plot with comprehensive unit circle guide
    theta1_mod = sol_coupled.y[0] % (2*np.pi)
    theta2_mod = sol_coupled.y[1] % (2*np.pi)
    ax_polar.plot(theta1_mod, np.ones_like(t_eval), 'r.', alpha=0.1)
    ax_polar.plot(theta2_mod, np.ones_like(t_eval), 'b.', alpha=0.1)

    # Update markers based on current time
    idx = np.abs(t_eval - t).argmin()
    theta1 = sol_coupled.y[0, idx] % (2*np.pi)
    theta2 = sol_coupled.y[1, idx] % (2*np.pi)

    polar_marker1, = ax_polar.plot([theta1], [1.0], 'ro', markersize=12, zorder=5)
    polar_marker2, = ax_polar.plot([theta2], [1.0], 'bo', markersize=12, zorder=5)

    # Update order parameter visualization
    r = r_coupled[idx]
    avg_phase = np.angle(np.exp(1j*sol_coupled.y[0, idx]) + np.exp(1j*sol_coupled.y[1, idx]))
    center_line, = ax_polar.plot([0, avg_phase], [0, r], 'm-', linewidth=2.5, alpha=0.7)
    ax_polar.text(0, 1.15, f'r = {r:.2f}', ha='center', fontsize=12, color='magenta')

    # Add unit circle reference points
    angles = [0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi]
    labels = ['0', 'π/2', 'π', '3π/2', '2π']
    colors = ['k', 'g', 'k', 'g', 'k']

    for i, (angle, label, color) in enumerate(zip(angles, labels, colors)):
        ax_polar.plot([angle, angle], [0, 1.1], color=color, linestyle=':', alpha=0.5)
        ax_polar.text(angle, 1.15, label, ha='center', color=color, fontsize=12)

    # Add signal value references
    ax_polar.text(0, 1.05, 'sin(θ)=0', ha='center', va='bottom', fontsize=10)
    ax_polar.text(np.pi/2, 1.05, 'sin(θ)=1', ha='center', va='bottom', fontsize=10)
    ax_polar.text(np.pi, 1.05, 'sin(θ)=0', ha='center', va='bottom', fontsize=10)
    ax_polar.text(3*np.pi/2, 1.05, 'sin(θ)=-1', ha='center', va='bottom', fontsize=10)

    ax_polar.set_title('Phase Space (Unit Circle)', fontsize=16, pad=20)
    ax_polar.set_rmax(1.2)
    ax_polar.set_rticks([0.5, 1.0])
    ax_polar.set_rlabel_position(0)

    # Add unit circle explanation panel
    plt.figtext(0.72, 0.28, "UNIT CIRCLE KEY:\n\n"
                "• Angle = Oscillator phase (θ)\n"
                "• Position = (cosθ, sinθ)\n"
                "• Full circle = 2π rad (1 cycle)\n\n"
                "SYNCHRONIZATION:\n"
                "• Clustered points = Synchronized\n"
                "• Spread points = Desynchronized\n\n"
                "ORDER PARAMETER (r):\n"
                "• Magenta line length\n"
                "• r=0: No sync (points spread)\n"
                "• r=1: Perfect sync (same point)\n\n"
                "PHASE-LOCKING:\n"
                "• Fixed angular distance\n"
                "• Same frequency, different phase",
                bbox=dict(facecolor='ivory', alpha=0.8, edgecolor='chocolate', boxstyle='round'),
                fontsize=11,
                horizontalalignment='left',
                verticalalignment='top')

    # Update markers on the main plots
    phase_marker1.set_data([t], [sol_coupled.y[0, idx]])
    phase_marker2.set_data([t], [sol_coupled.y[1, idx]])
    signal_marker1.set_data([t], [signal_coupled[0, idx]])
    signal_marker2.set_data([t], [signal_coupled[1, idx]])

    plt.tight_layout(rect=[0, 0, 1, 0.95])

    # Return the figure
    return fig

# %% [code]
# Function to update visualization based on time
def update_visualization(change):
    t = change['new']

    # Clear previous output
    with main_output:
        clear_output(wait=True)

        # Create and display the visualization
        fig = create_visualization(t)

        # Determine synchronization status
        idx = np.abs(t_eval - t).argmin()
        r = r_coupled[idx]

        if r > 0.8:
            sync_status = "✅ STRONG SYNCHRONIZATION"
            sync_color = "green"
        elif r > 0.4:
            sync_status = "🔄 PARTIAL SYNCHRONIZATION"
            sync_color = "orange"
        else:
            sync_status = "❌ NO SYNCHRONIZATION"
            sync_color = "red"

        # Educational insights based on time
        if t < 1.0:
            insight = "INITIAL CONDITION: Oscillators starting at same phase but different frequencies"
        elif t < 2.5:
            insight = "SYNCHRONIZATION PROCESS: Coupled oscillators beginning to align frequencies"
        elif t < 4.0:
            insight = "PHASE-LOCKING: Oscillators maintaining fixed phase relationship"
        else:
            insight = "STEADY STATE: Partial synchronization achieved (K < K_critical)"

        phase1 = sol_coupled.y[0, idx] % (2*np.pi)
        phase2 = sol_coupled.y[1, idx] % (2*np.pi)
        phase_diff = (phase1 - phase2) % (2*np.pi)
        if phase_diff > np.pi:
            phase_diff = 2*np.pi - phase_diff

        # Display educational information
        print(f"\nTIME: {t:.2f}s | {sync_status}")
        print(f"Oscillator 1: θ = {sol_coupled.y[0, idx]:.2f} rad ({np.degrees(phase1):.1f}°) | sin(θ) = {signal_coupled[0, idx]:.2f}")
        print(f"Oscillator 2: θ = {sol_coupled.y[1, idx]:.2f} rad ({np.degrees(phase2):.1f}°) | sin(θ) = {signal_coupled[1, idx]:.2f}")
        print(f"Phase difference: {phase_diff:.2f} rad | Order parameter: r = {r:.2f}")
        print(f"\n💡 {insight}")

        # Display the figure
        plt.show(fig)
        plt.close(fig)

# %% [code]
# Set up the event listener
time_slider.observe(update_visualization, names='value')

# Create initial visualization
initial_fig = create_visualization(0)

# Display the timeline controls first
print("="*80)
print("🎛️ DEDICATED TIMELINE CONTROL PANEL")
print("="*80)
print("• Use the slider below to navigate through time")
print("• Scroll wheel works on the slider for precise control")
print("• Click anywhere on the slider for instant time jumps")
print("• Use play button for automatic animation")
print("="*80)

# Display the timeline controls
display(widgets.VBox([
    widgets.HBox([time_slider, play_button]),
    widgets.Label("🖱️ TIPS: Scroll wheel works on slider | Click for precise positioning | Right-click slider for menu")
]))

# Display the initial visualization
with main_output:
    print("\nTIME: 0.00s | ❌ NO SYNCHRONIZATION")
    print("Oscillator 1: θ = 0.00 rad (0.0°) | sin(θ) = 0.00")
    print("Oscillator 2: θ = 0.00 rad (0.0°) | sin(θ) = 0.00")
    print("Phase difference: 0.00 rad | Order parameter: r = 1.00")
    print("\n💡 INITIAL CONDITION: Oscillators starting at same phase but different frequencies")
    plt.show(initial_fig)

# Display the main output
display(main_output)

🎛️ DEDICATED TIMELINE CONTROL PANEL
• Use the slider below to navigate through time
• Scroll wheel works on the slider for precise control
• Click anywhere on the slider for instant time jumps
• Use play button for automatic animation


Output()

In [16]:
print("\n" + "="*80)
print("🚀 KURAMOTO OSCILLATOR EXPLORATION - DEDICATED CONTROL GUIDE")
print("="*80)

print("\n🎛️ USING THE DEDICATED TIMELINE CONTROLS:")
print("• The timeline control is COMPLETELY SEPARATE from the visualization")
print("• It appears ABOVE the visualization in its own panel")
print("• No UI elements are incorporated into the main image")
print("• All navigation happens through the dedicated slider and play button")

print("\n🔍 STEP 1: INITIAL STATE EXPLORATION (t = 0-0.5s)")
print("• Drag the slider to t = 0.2s or use play button to advance slowly")
print("• Observe: Both oscillators start at phase 0 but have different slopes")
print("• Key insight: Different frequencies cause immediate divergence")
print("• Check unit circle: Points begin to separate immediately")

print("\n🔍 STEP 2: SYNCHRONIZATION PROCESS (t = 1.0-2.0s)")
print("• Move slider to t = 1.5s")
print("• Watch: Coupled oscillators' phases begin converging (lines getting closer)")
print("• Check unit circle: Points moving toward each other (increasing r value)")
print("• Real-world connection: Like fireflies adjusting flash timing to neighbors")

print("\n🔍 STEP 3: PHASE-LOCKING OBSERVATION (t = 2.5-3.5s)")
print("• Position slider at t = 3.0s")
print("• Critical observation: Phase difference becomes constant (parallel lines)")
print("• Unit circle: Fixed angular distance between points")
print("• This is 'phase-locking' - same frequency, different phase")
print("• Biological example: Heart pacemaker cells maintaining rhythm")

print("\n🔍 STEP 4: STEADY STATE ANALYSIS (t = 4.0-5.0s)")
print("• Move to t = 4.5s")
print("• Coupled system: Phases maintain constant difference")
print("• Uncoupled system: Phase difference keeps growing")
print("• Signal view: Coupled signals maintain alignment")
print("• Engineering application: Power grid stability requires this")

print("\n💡 PRO TIPS FOR DEDICATED CONTROLS:")
print("1. Use the play button for automatic animation of the process")
print("2. Right-click the slider for additional options (reset, etc.)")
print("3. Scroll wheel on the slider for precise time control")
print("4. Click directly on slider for instant time jumps")
print("5. The visualization remains clean with no embedded UI elements")

print("\n🎯 REAL-WORLD CONNECTIONS TO OBSERVE:")
print("• Firefly synchronization: When r approaches 1, all flash together")
print("• Power grids: Must maintain r ≈ 1 for stable operation")
print("• Brain waves: Different r values correspond to different cognitive states")
print("• Cardiac cells: Phase-locking enables coordinated heart contractions")

print("\n" + "="*80)
print("🔑 KEY TO UNIT CIRCLE INTERPRETATION:")
print("="*80)
print("• Position angle = Oscillator phase (θ)")
print("• Angular velocity = Frequency (dθ/dt)")
print("• Cluster tightness = Synchronization level (r)")
print("• Center of mass position = Average phase")
print("• Movement toward clustering = Synchronization process")
print("• Stable cluster = Phase-locked state")
print("• Even distribution = Complete desynchronization")
print("="*80)


🚀 KURAMOTO OSCILLATOR EXPLORATION - DEDICATED CONTROL GUIDE

🎛️ USING THE DEDICATED TIMELINE CONTROLS:
• The timeline control is COMPLETELY SEPARATE from the visualization
• It appears ABOVE the visualization in its own panel
• No UI elements are incorporated into the main image
• All navigation happens through the dedicated slider and play button

🔍 STEP 1: INITIAL STATE EXPLORATION (t = 0-0.5s)
• Drag the slider to t = 0.2s or use play button to advance slowly
• Observe: Both oscillators start at phase 0 but have different slopes
• Key insight: Different frequencies cause immediate divergence
• Check unit circle: Points begin to separate immediately

🔍 STEP 2: SYNCHRONIZATION PROCESS (t = 1.0-2.0s)
• Move slider to t = 1.5s
• Watch: Coupled oscillators' phases begin converging (lines getting closer)
• Check unit circle: Points moving toward each other (increasing r value)
• Real-world connection: Like fireflies adjusting flash timing to neighbors

🔍 STEP 3: PHASE-LOCKING OBSERVATION